In [1]:
import json
from pathlib import Path

input_path = Path("data/raw/natural_hallu/03_natural_generated.json")
output_path = Path("data/raw/natural_hallu/03_natural_generated_filtered.json")

keep_fields = [
    "chunk_id",
    "doc_id",
    "domain",
    "content",
    "question_type",
    "question",
    "model_response",
]

with input_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

filtered_data = [
    {field: sample.get(field) for field in keep_fields}
    for sample in data
]

with output_path.open("w", encoding="utf-8") as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)

print(f"Đã xử lý {len(filtered_data):,} mẫu.")
print(f"File đầu ra: {output_path}")

Đã xử lý 14,882 mẫu.
File đầu ra: data/raw/natural_hallu/03_natural_generated_filtered.json


# grouth truth

In [2]:
import json
from pathlib import Path

input_path = Path("data/raw/natural_hallu/03_natural_generated.json")
output_path = Path("data/raw/natural_hallu/ground_truth_answer.json")

keep_fields = [
    "chunk_id",
    "doc_id",
    "domain",
    "content",
    "question_type",
    "question",
    "answer",
]

with input_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

filtered_data = [
    {field: sample.get(field) for field in keep_fields}
    for sample in data
]

with output_path.open("w", encoding="utf-8") as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)

print(f"Đã xử lý {len(filtered_data):,} mẫu.")
print(f"File đầu ra: {output_path}")

Đã xử lý 14,882 mẫu.
File đầu ra: data/raw/natural_hallu/ground_truth_answer.json


# mẫu judge 

In [1]:
# -*- coding: utf-8 -*-

"""
Tạo tập demo 30 mẫu để chạy LLM Judge.

Nguồn:
1. Ground-truth answer do LLM sinh:
   - label gốc: no_hallucination

2. Guided hallucinated answer:
   - DeepSeek
   - Qwen
   - label gốc: intrinsic / extrinsic / mixed

Output chỉ giữ:
- chunk_id
- doc_id
- domain
- content
- question
- answer
- label

Lưu ý:
'label' là nhãn gốc cần được Judge kiểm định lại,
không phải nhãn chắc chắn chính xác.
"""

import json
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd


# ============================================================
# 1. CONFIG
# ============================================================

DATA_DIR = Path("data")

GROUND_TRUTH_FILE = (
    DATA_DIR
    / "grouth_truth_answer"
    / "ground_truth_answer.json"
)

DEEPSEEK_FILE = (
    DATA_DIR
    / "guided_hallu_qa"
    / "deepseek_clean.json"
)

QWEN_FILE = (
    DATA_DIR
    / "guided_hallu_qa"
    / "qwen_clean.json"
)

OUTPUT_JSON = (
    DATA_DIR
    / "judge_demo_30.json"
)

OUTPUT_CSV = (
    DATA_DIR
    / "judge_demo_30.csv"
)

RANDOM_SEED = 42

# Tổng cộng 30 mẫu
LABEL_QUOTAS = {
    "no_hallucination": 8,
    "intrinsic_hallucination": 8,
    "extrinsic_hallucination": 7,
    "mixed_hallucination": 7,
}


# ============================================================
# 2. HELPERS
# ============================================================

def load_json(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file: {path.resolve()}"
        )

    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        data = json.load(file)

    if not isinstance(data, list):
        raise ValueError(
            f"{path} không phải JSON list."
        )

    return data


def save_json(data, path):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )


def clean_value(value, default=None):
    if value is None:
        return default

    if isinstance(value, float) and pd.isna(value):
        return default

    if isinstance(value, str):
        value = value.strip()

        if not value:
            return default

    return value


def normalize_text(value):
    if value is None:
        return ""

    value = str(value).strip().lower()

    return re.sub(
        r"\s+",
        " ",
        value,
    )


def normalize_label(label):
    """
    Chuẩn hóa các biến thể nhãn về bốn nhãn thống nhất.
    """

    if label is None:
        return None

    label = str(label).strip().lower()

    label = label.replace("-", "_")
    label = label.replace(" ", "_")

    mapping = {
        # No hallucination
        "no_hallu": "no_hallucination",
        "no_hallucination": "no_hallucination",
        "non_hallucination": "no_hallucination",
        "supported": "no_hallucination",

        # Intrinsic
        "intrinsic": "intrinsic_hallucination",
        "intrinsic_hallu": "intrinsic_hallucination",
        "intrinsic_hallucination": "intrinsic_hallucination",

        # Extrinsic
        "extrinsic": "extrinsic_hallucination",
        "extrinsic_hallu": "extrinsic_hallucination",
        "extrinsic_hallucination": "extrinsic_hallucination",

        # Mixed
        "mixed": "mixed_hallucination",
        "mixed_hallu": "mixed_hallucination",
        "mixed_partial": "mixed_hallucination",
        "mixed_hallucination": "mixed_hallucination",
    }

    return mapping.get(
        label,
        label,
    )


# ============================================================
# 3. LOAD FILES
# ============================================================

ground_truth_data = load_json(
    GROUND_TRUTH_FILE
)

deepseek_data = load_json(
    DEEPSEEK_FILE
)

qwen_data = load_json(
    QWEN_FILE
)

print("=" * 80)
print("INPUT DATA")
print("=" * 80)

print(
    "Ground truth:",
    len(ground_truth_data),
)

print(
    "DeepSeek:",
    len(deepseek_data),
)

print(
    "Qwen:",
    len(qwen_data),
)


# ============================================================
# 4. BUILD GROUND-TRUTH INDEX
# ============================================================

ground_truth_by_exact_key = {}

ground_truth_by_chunk = defaultdict(
    list
)

for row in ground_truth_data:
    chunk_id = clean_value(
        row.get("chunk_id")
    )

    question = clean_value(
        row.get("question")
    )

    if chunk_id is None:
        continue

    exact_key = (
        chunk_id,
        normalize_text(question),
    )

    ground_truth_by_exact_key[
        exact_key
    ] = row

    ground_truth_by_chunk[
        chunk_id
    ].append(row)


def find_ground_truth(sample):
    """
    Tìm mẫu ground truth tương ứng.

    Ưu tiên:
    1. chunk_id + question
    2. chunk_id nếu chunk đó chỉ có một câu hỏi
    """

    chunk_id = clean_value(
        sample.get("chunk_id")
    )

    question = clean_value(
        sample.get("question")
    )

    if chunk_id is None:
        return None

    if question is not None:
        exact_key = (
            chunk_id,
            normalize_text(question),
        )

        matched = ground_truth_by_exact_key.get(
            exact_key
        )

        if matched is not None:
            return matched

    candidates = ground_truth_by_chunk.get(
        chunk_id,
        [],
    )

    if len(candidates) == 1:
        return candidates[0]

    return None


# ============================================================
# 5. STANDARDIZE NO-HALLUCINATION SAMPLES
# ============================================================

def standardize_ground_truth(row):
    answer = clean_value(
        row.get("answer")
    )

    if answer is None:
        answer = clean_value(
            row.get("gold_answer")
        )

    sample = {
        "chunk_id": clean_value(
            row.get("chunk_id")
        ),
        "doc_id": clean_value(
            row.get("doc_id")
        ),
        "domain": clean_value(
            row.get("domain"),
            "unknown",
        ),
        "content": clean_value(
            row.get("content")
        ),
        "question": clean_value(
            row.get("question")
        ),
        "answer": answer,
        "label": "no_hallucination",

        # Chỉ dùng khi chọn mẫu, không ghi ra output.
        "_question_type": clean_value(
            row.get("question_type"),
            "unknown",
        ),
        "_generator": "ground_truth_generator",
    }

    required_fields = [
        "chunk_id",
        "doc_id",
        "content",
        "question",
        "answer",
    ]

    missing_fields = [
        field
        for field in required_fields
        if not sample.get(field)
    ]

    if missing_fields:
        return None

    return sample


no_hallucination_candidates = []

for row in ground_truth_data:
    sample = standardize_ground_truth(
        row
    )

    if sample is not None:
        no_hallucination_candidates.append(
            sample
        )


# ============================================================
# 6. STANDARDIZE GUIDED HALLUCINATION SAMPLES
# ============================================================

def get_hallucinated_answer(row):
    """
    Hỗ trợ nhiều tên trường khác nhau.
    """

    possible_fields = [
        "hallu_answer",
        "hallucinated_answer",
        "model_response",
        "generated_answer",
        "response",
    ]

    for field in possible_fields:
        value = clean_value(
            row.get(field)
        )

        if value is not None:
            return value

    return None


def get_target_label(row):
    """
    Hỗ trợ nhiều tên trường nhãn khác nhau.
    """

    possible_fields = [
        "target_label",
        "target_hallucination_label",
        "requested_label",
        "label",
    ]

    for field in possible_fields:
        value = clean_value(
            row.get(field)
        )

        if value is not None:
            return normalize_label(value)

    return None


def standardize_guided(row, generator):
    matched_ground_truth = find_ground_truth(
        row
    )

    chunk_id = clean_value(
        row.get("chunk_id")
    )

    doc_id = clean_value(
        row.get("doc_id")
    )

    domain = clean_value(
        row.get("domain")
    )

    content = clean_value(
        row.get("content")
    )

    question = clean_value(
        row.get("question")
    )

    question_type = clean_value(
        row.get("question_type")
    )

    answer = get_hallucinated_answer(
        row
    )

    label = get_target_label(
        row
    )

    # Bổ sung trường còn thiếu từ ground-truth data.
    if matched_ground_truth is not None:
        if doc_id is None:
            doc_id = clean_value(
                matched_ground_truth.get("doc_id")
            )

        if domain is None:
            domain = clean_value(
                matched_ground_truth.get("domain")
            )

        if content is None:
            content = clean_value(
                matched_ground_truth.get("content")
            )

        if question is None:
            question = clean_value(
                matched_ground_truth.get("question")
            )

        if question_type is None:
            question_type = clean_value(
                matched_ground_truth.get(
                    "question_type"
                )
            )

    sample = {
        "chunk_id": chunk_id,
        "doc_id": doc_id,
        "domain": domain or "unknown",
        "content": content,
        "question": question,
        "answer": answer,
        "label": label,

        # Chỉ dùng để lấy mẫu đa dạng.
        "_question_type": (
            question_type or "unknown"
        ),
        "_generator": generator,
    }

    required_fields = [
        "chunk_id",
        "doc_id",
        "content",
        "question",
        "answer",
        "label",
    ]

    missing_fields = [
        field
        for field in required_fields
        if not sample.get(field)
    ]

    if missing_fields:
        return None, missing_fields

    if label not in {
        "intrinsic_hallucination",
        "extrinsic_hallucination",
        "mixed_hallucination",
    }:
        return None, [
            f"unsupported_label:{label}"
        ]

    return sample, []


guided_candidates = []

invalid_guided = []

for generator, rows in [
    ("deepseek", deepseek_data),
    ("qwen", qwen_data),
]:
    for row_index, row in enumerate(rows):
        sample, errors = standardize_guided(
            row,
            generator,
        )

        if sample is None:
            invalid_guided.append({
                "generator": generator,
                "row_index": row_index,
                "chunk_id": row.get("chunk_id"),
                "errors": errors,
            })

            continue

        guided_candidates.append(
            sample
        )


# ============================================================
# 7. REMOVE DUPLICATES
# ============================================================

def remove_duplicates(samples):
    unique_samples = []

    seen = set()

    for sample in samples:
        key = (
            sample.get("label"),
            sample.get("chunk_id"),
            normalize_text(
                sample.get("question")
            ),
            normalize_text(
                sample.get("answer")
            ),
        )

        if key in seen:
            continue

        seen.add(key)

        unique_samples.append(
            sample
        )

    return unique_samples


no_hallucination_candidates = (
    remove_duplicates(
        no_hallucination_candidates
    )
)

guided_candidates = remove_duplicates(
    guided_candidates
)

print("\n" + "=" * 80)
print("VALID CANDIDATES")
print("=" * 80)

print(
    "No hallucination:",
    len(no_hallucination_candidates),
)

print(
    "Guided hallucination:",
    len(guided_candidates),
)

print(
    "Invalid guided:",
    len(invalid_guided),
)


# ============================================================
# 8. GROUP CANDIDATES BY LABEL
# ============================================================

candidates_by_label = defaultdict(
    list
)

for sample in no_hallucination_candidates:
    candidates_by_label[
        sample["label"]
    ].append(sample)

for sample in guided_candidates:
    candidates_by_label[
        sample["label"]
    ].append(sample)

print("\nAvailable samples by label:")

for label, candidates in candidates_by_label.items():
    generator_counts = Counter(
        sample["_generator"]
        for sample in candidates
    )

    print(
        f"{label:28s}: "
        f"{len(candidates):6d} | "
        f"{dict(generator_counts)}"
    )


# ============================================================
# 9. DIVERSITY-AWARE SAMPLING
# ============================================================

def diversity_sample(
    candidates,
    sample_size,
    seed,
):
    """
    Ưu tiên:
    - domain chưa được chọn;
    - question type chưa được chọn;
    - generator chưa được chọn;
    - chunk_id chưa được chọn.
    """

    if len(candidates) < sample_size:
        raise ValueError(
            f"Không đủ mẫu: cần {sample_size}, "
            f"nhưng chỉ có {len(candidates)}."
        )

    rng = random.Random(seed)

    remaining = list(candidates)

    rng.shuffle(remaining)

    selected = []

    domain_counts = Counter()
    question_type_counts = Counter()
    generator_counts = Counter()
    chunk_counts = Counter()

    while len(selected) < sample_size:
        best_score = None
        best_indices = []

        for index, sample in enumerate(remaining):
            domain = sample.get(
                "domain",
                "unknown",
            )

            question_type = sample.get(
                "_question_type",
                "unknown",
            )

            generator = sample.get(
                "_generator",
                "unknown",
            )

            chunk_id = sample.get(
                "chunk_id",
                "unknown",
            )

            score = (
                5.0 / (
                    1 + domain_counts[domain]
                )
                + 5.0 / (
                    1
                    + question_type_counts[
                        question_type
                    ]
                )
                + 4.0 / (
                    1
                    + generator_counts[generator]
                )
                + 3.0 / (
                    1
                    + chunk_counts[chunk_id]
                )
            )

            if (
                best_score is None
                or score > best_score
            ):
                best_score = score
                best_indices = [index]

            elif score == best_score:
                best_indices.append(index)

        chosen_index = rng.choice(
            best_indices
        )

        chosen = remaining.pop(
            chosen_index
        )

        selected.append(
            chosen
        )

        domain_counts[
            chosen.get("domain", "unknown")
        ] += 1

        question_type_counts[
            chosen.get(
                "_question_type",
                "unknown",
            )
        ] += 1

        generator_counts[
            chosen.get(
                "_generator",
                "unknown",
            )
        ] += 1

        chunk_counts[
            chosen.get(
                "chunk_id",
                "unknown",
            )
        ] += 1

    return selected


# ============================================================
# 10. SELECT 30 SAMPLES
# ============================================================

selected_samples = []

for label_index, (
    label,
    quota,
) in enumerate(
    LABEL_QUOTAS.items()
):
    label_candidates = candidates_by_label[
        label
    ]

    selected = diversity_sample(
        candidates=label_candidates,
        sample_size=quota,
        seed=RANDOM_SEED + label_index,
    )

    selected_samples.extend(
        selected
    )


# ============================================================
# 11. SHUFFLE AND KEEP ONLY REQUIRED FIELDS
# ============================================================

random.Random(
    RANDOM_SEED
).shuffle(
    selected_samples
)

OUTPUT_FIELDS = [
    "chunk_id",
    "doc_id",
    "domain",
    "content",
    "question",
    "answer",
    "label",
]

final_samples = []

for sample in selected_samples:
    final_samples.append({
        field: sample.get(field)
        for field in OUTPUT_FIELDS
    })


# ============================================================
# 12. VALIDATE OUTPUT
# ============================================================

expected_total = sum(
    LABEL_QUOTAS.values()
)

if len(final_samples) != expected_total:
    raise ValueError(
        f"Output có {len(final_samples)} mẫu, "
        f"không đúng {expected_total} mẫu."
    )

for index, sample in enumerate(final_samples):
    missing_fields = [
        field
        for field in OUTPUT_FIELDS
        if sample.get(field) is None
    ]

    if missing_fields:
        raise ValueError(
            f"Mẫu {index} thiếu: {missing_fields}"
        )


# ============================================================
# 13. SAVE OUTPUT
# ============================================================

save_json(
    final_samples,
    OUTPUT_JSON,
)

output_df = pd.DataFrame(
    final_samples
)

output_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 14. REPORT
# ============================================================

print("\n" + "=" * 80)
print("FINAL DEMO DATASET")
print("=" * 80)

print(
    "Total samples:",
    len(final_samples),
)

print("\nLabel distribution:")

print(
    output_df[
        "label"
    ].value_counts()
)

print("\nDomain distribution:")

print(
    output_df[
        "domain"
    ].value_counts()
)

print("\nLabel × Domain:")

print(
    pd.crosstab(
        output_df["label"],
        output_df["domain"],
        margins=True,
    )
)

print("\nOutput JSON:")
print(
    OUTPUT_JSON.resolve()
)

print("\nOutput CSV:")
print(
    OUTPUT_CSV.resolve()
)

display(
    output_df[
        [
            "chunk_id",
            "domain",
            "question",
            "answer",
            "label",
        ]
    ]
)

INPUT DATA
Ground truth: 14882
DeepSeek: 13109
Qwen: 595

VALID CANDIDATES
No hallucination: 14882
Guided hallucination: 13704
Invalid guided: 0

Available samples by label:
no_hallucination            :  14882 | {'ground_truth_generator': 14882}
mixed_hallucination         :   2180 | {'deepseek': 2089, 'qwen': 91}
intrinsic_hallucination     :   5627 | {'deepseek': 5368, 'qwen': 259}
extrinsic_hallucination     :   5897 | {'deepseek': 5652, 'qwen': 245}

FINAL DEMO DATASET
Total samples: 30

Label distribution:
label
intrinsic_hallucination    8
no_hallucination           8
extrinsic_hallucination    7
mixed_hallucination        7
Name: count, dtype: int64

Domain distribution:
domain
education    8
legal        5
news         5
finance      4
medical      4
science      4
Name: count, dtype: int64

Label × Domain:
domain                   education  finance  legal  medical  news  science  \
label                                                                        
extrinsic_halluc

,chunk_id,domain,question,answer,label
0,finance_sbv_22407_chunk2,finance,Những yếu tố nào về tình hình trên địa bàn tỉn...,Đồng chí Nguyễn Phi Long đã trao đổi về tình h...,extrinsic_hallucination
1,education_uit_90454_chunk16,education,Trong tình huống bạn Hoàng Thị Yến đặt câu hỏi...,Yêu cầu trang bị kiến thức và bảo hộ đối với s...,intrinsic_hallucination
2,finance_sbv_29646_chunk8,finance,Việc rèn luyện đạo đức và trách nhiệm nêu gươn...,"Việc rèn luyện, tu dưỡng đạo đức gắn với trách...",intrinsic_hallucination
3,finance_sbv_90553_chunk6,finance,Huy hiệu 30 năm tuổi Đảng mang ý nghĩa gì đối ...,"Đối với đồng chí Phạm Duy Bình, Huy hiệu 30 nă...",mixed_hallucination
4,legal_congbao_20220_chunk100,legal,"Sau khi bán đấu giá, khoản tiền thu về sẽ được...",Sau khi trừ các chi phí hợp lý của các bên liê...,extrinsic_hallucination
5,legal_congbao_00576_chunk34,legal,Số lượng mẫu và loại tài liệu cần có trong hồ ...,Hồ sơ đăng ký gồm 2 loại tài liệu: 1 đơn đề ng...,no_hallucination
6,finance_vtv_money_78970_chunk13,finance,Tại sao sự dịch chuyển chuỗi cung ứng lại được...,Vì các doanh nghiệp toàn cầu đang tái thiết ch...,no_hallucination
7,legal_congbao_08140_chunk108,legal,Ai là cơ quan đại diện chủ sở hữu đối với các ...,Cơ quan đại diện chủ sở hữu là cơ quan quản lý...,intrinsic_hallucination
8,medical_suckhoedoisong_76253_chunk5,medical,Mục đích của việc tư vấn người bệnh tuân thủ đ...,Mục đích của việc tư vấn người bệnh tuân thủ đ...,intrinsic_hallucination
9,news_vnexpress_95730_chunk4,news,Phó chủ tịch Hạ viện Ba Lan Krzysztof Bosak đã...,Ông yêu cầu thông qua luật cấm chuyển giao vũ ...,intrinsic_hallucination


# check file

In [1]:
import pandas as pd

df = pd.read_json("data/output/judge2/judge2_v4_merged_final.json")
df.head()

,qa_id,chunk_id,doc_id,domain,context,question,answer,original_label,judge,model,batch_id,status,evaluation,usage,provider,repaired_from_raw_output,error,raw_output
0,medical_suckhoedoisong_18503_chunk19__qa1,medical_suckhoedoisong_18503_chunk19,medical_suckhoedoisong_18503,medical,Viêm gan B là bệnh lý nguy hiểm nếu không phát...,Tiêm vaccine phòng viêm gan B được xem là phươ...,Tiêm vaccine phòng viêm gan B được xem là phươ...,intrinsic_hallucination,judge2,openai/gpt-5.6-luna-pro,batch-1790084655-lhxiUr2KYXoxyDSJo1iq,success,"{'actual_label': 'extrinsic_hallucination', 'l...","{'prompt_tokens': 19955, 'completion_tokens': ...",NaN,NaN,NaN,NaN
1,science_vjst_33219_chunk6__qa1,science_vjst_33219_chunk6,science_vjst_33219,science,Các axit này là sản phẩm chính của quá trình o...,Tại sao việc hiểu rõ thành phần của các axit h...,Việc hiểu rõ thành phần của các axit hữu cơ mạ...,mixed_hallucination,judge2,openai/gpt-5.6-luna-pro,batch-1790084655-lhxiUr2KYXoxyDSJo1iq,success,"{'actual_label': 'extrinsic_hallucination', 'l...","{'prompt_tokens': 21462, 'completion_tokens': ...",NaN,NaN,NaN,NaN
2,finance_sbv_46927_chunk4__qa1,finance_sbv_46927_chunk4,finance_sbv_46927,finance,"T hời gian tới, bên cạnh việc chỉ đạo các ngân...",Ngành Ngân hàng sẽ tập trung đặc biệt vào công...,Ngành Ngân hàng sẽ tập trung đặc biệt vào công...,no_hallucination,judge2,openai/gpt-5.6-luna-pro,batch-1790084655-lhxiUr2KYXoxyDSJo1iq,success,"{'actual_label': 'intrinsic_hallucination', 'l...","{'prompt_tokens': 21064, 'completion_tokens': ...",NaN,NaN,NaN,NaN
3,finance_sbv_60691_chunk28__qa1,finance_sbv_60691_chunk28,finance_sbv_60691,finance,Thực hiện nghiêm Kết luận của Hội nghị Trung ư...,"Nhằm mục đích gì, Đảng và hệ thống chính trị c...",Đảng và hệ thống chính trị cần được xây dựng v...,intrinsic_hallucination,judge2,openai/gpt-5.6-luna-pro,batch-1790084655-lhxiUr2KYXoxyDSJo1iq,success,"{'actual_label': 'extrinsic_hallucination', 'l...","{'prompt_tokens': 19712, 'completion_tokens': ...",NaN,NaN,NaN,NaN
4,legal_vbpl_12745_chunk11__qa1,legal_vbpl_12745_chunk11,legal_vbpl_12745,legal,2. Cơ quan bảo hiểm xã hội có trách nhiệm triể...,Khi phát sinh thay đổi về điều kiện kinh tế - ...,Khi có thay đổi về điều kiện kinh tế - xã hội ...,extrinsic_hallucination,judge2,openai/gpt-5.6-luna-pro,batch-1790084655-lhxiUr2KYXoxyDSJo1iq,success,"{'actual_label': 'no_hallucination', 'label_ma...","{'prompt_tokens': 19771, 'completion_tokens': ...",NaN,NaN,NaN,NaN


In [3]:
df["evaluation"].apply(
    lambda x: x.get("confidence") if isinstance(x, dict) else None
).value_counts(dropna=False).sort_index()

evaluation
0.76      1
0.78     82
0.79      1
0.80     48
0.82    520
0.83     15
0.84    199
0.88      5
0.89      2
0.90     37
0.91     54
0.92     34
0.93    134
0.94    131
0.95    212
0.96    462
0.97    143
0.98    545
0.99    432
NaN       5
Name: count, dtype: int64

In [11]:
import json

with open("demo.json", "r", encoding="utf-8") as f:
    try:
        data = json.load(f)
        print("JSON hợp lệ")
    except json.JSONDecodeError as e:
        print("JSON lỗi:")
        print(e)

JSON lỗi:
Expecting value: line 20374 column 5 (char 982794)


In [13]:
with open("demo.json", "r", encoding="utf-8") as f:
    lines = f.readlines()

for i in range(20369, 20379):
    print(f"{i+1}: {lines[i]!r}")

20370: '      "self_contradiction": false,\n'
20371: '      "short_reason": "CONTEXT chỉ nêu CO2 và H2S bị giữ trong dung dịch, H2S dư được hấp phụ, không nêu việc loại bỏ hoàn toàn hay chuyển H2S thành lưu huỳnh."\n'
20372: '    },\n'
20373: '    "human_label": "mixed_hallucination"\n'
20374: '  },'


IndexError: list index out of range

In [1]:
import pandas as pd
df = pd.read_json("data/output/judge2_human_review_v4_quality_filtered.json")
len(df)

target = "finance_sbv_82873_chunk4__qa1"

idx = df.index[df["qa_id"] == target]

print(idx)

Index([462], dtype='int64')


In [1]:
import pandas as pd 

df = pd.read_json("data/output/human_review/review/human_review_v4_quality_filtered.json")
df.head()

,qa_id,chunk_id,doc_id,domain,content,question,answer,original_label,original_hallucination_spans,judge1,judge2,human_label,human_completed,human_decision
0,medical_suckhoedoisong_18503_chunk19__qa1,medical_suckhoedoisong_18503_chunk19,medical_suckhoedoisong_18503,medical,Viêm gan B là bệnh lý nguy hiểm nếu không phát...,Tiêm vaccine phòng viêm gan B được xem là phươ...,Tiêm vaccine phòng viêm gan B được xem là phươ...,intrinsic_hallucination,[duy nhất],"{'model': 'google/gemma-4-31b-it', 'label': 'e...","{'model': 'openai/gpt-5.6-luna-pro', 'label': ...",extrinsic_hallucination,1.0,label
1,science_vjst_33219_chunk6__qa1,science_vjst_33219_chunk6,science_vjst_33219,science,Các axit này là sản phẩm chính của quá trình o...,Tại sao việc hiểu rõ thành phần của các axit h...,Việc hiểu rõ thành phần của các axit hữu cơ mạ...,mixed_hallucination,"[ăn mòn và đóng cặn kim phun nhiên liệu, kéo d...","{'model': 'google/gemma-4-31b-it', 'label': 'e...","{'model': 'openai/gpt-5.6-luna-pro', 'label': ...",extrinsic_hallucination,1.0,label
2,finance_sbv_46927_chunk4__qa1,finance_sbv_46927_chunk4,finance_sbv_46927,finance,"T hời gian tới, bên cạnh việc chỉ đạo các ngân...",Ngành Ngân hàng sẽ tập trung đặc biệt vào công...,Ngành Ngân hàng sẽ tập trung đặc biệt vào công...,no_hallucination,[],"{'model': 'google/gemma-4-31b-it', 'label': 'e...","{'model': 'openai/gpt-5.6-luna-pro', 'label': ...",intrinsic_hallucination,1.0,label
3,finance_sbv_60691_chunk28__qa1,finance_sbv_60691_chunk28,finance_sbv_60691,finance,Thực hiện nghiêm Kết luận của Hội nghị Trung ư...,"Nhằm mục đích gì, Đảng và hệ thống chính trị c...",Đảng và hệ thống chính trị cần được xây dựng v...,intrinsic_hallucination,"[trong sạch, vững mạnh, toàn diện]","{'model': 'google/gemma-4-31b-it', 'label': 'e...","{'model': 'openai/gpt-5.6-luna-pro', 'label': ...",extrinsic_hallucination,1.0,label
4,news_tuoitre_04102_chunk0__qa1,news_tuoitre_04102_chunk0,news_tuoitre_04102,news,Ông Darren Chua từng sống và làm việc ở TP.HCM...,Ông Darren Chua hiện đang sinh sống và làm việ...,Ông Darren Chua hiện đang sinh sống và làm việ...,extrinsic_hallucination,[Singapore],"{'model': 'google/gemma-4-31b-it', 'label': 'i...","{'model': 'openai/gpt-5.6-luna-pro', 'label': ...",intrinsic_hallucination,1.0,label
